In [1]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
# import joblib
# import numpy_financial as npf

# Matplotlib darkmode
plt.style.use('dark_background')

# display settings
pd.set_option('display.max_columns', None)

In [2]:
import glob

path = f"../data/2_silver_cleaned/munis=*/type=rent/master_rentals.parquet"
files = glob.glob(path)

df = pd.concat([pd.read_parquet(f) for f in files])

df_raw = df.copy()

In [3]:
df.head()

,listing_number,scrape_at,price,title,type,suburb,city,munis,bedrooms,bathrooms,parking,floor_size,is_estate,has_security,is_modern
0,116640288,2026-02-16 05:34:24.414485,7000.0,2 Bedroom House,House,fochville,fochville,West Rand,2.0,1.0,2.0,NaN,0,0,0
1,116787840,2026-02-16 05:34:24.415094,3650.0,2 Bedroom Apartment,Apartment,fochville,fochville,West Rand,2.0,NaN,1.0,NaN,0,0,1
2,113833369,2026-02-16 05:34:24.415590,4700.0,1 Bedroom Apartment,Apartment,fochville,fochville,West Rand,1.0,2.0,1.0,NaN,0,0,1
3,113521107,2026-02-16 05:34:24.416130,3650.0,2 Bedroom Apartment,Apartment,fochville,fochville,West Rand,2.0,NaN,1.0,NaN,0,0,1
4,116838798,2026-02-16 10:52:36.920156,11000.0,3 Bedroom Apartment,Apartment,agavia,krugersdorp,West Rand,3.0,2.0,2.0,NaN,0,0,0


In [84]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20413 entries, 0 to 338
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   listing_number  20413 non-null  object        
 1   scrape_at       20413 non-null  datetime64[ns]
 2   price           20370 non-null  float64       
 3   title           19551 non-null  object        
 4   type            19551 non-null  object        
 5   suburb          20413 non-null  object        
 6   city            20413 non-null  object        
 7   munis           20413 non-null  object        
 8   bedrooms        19634 non-null  float64       
 9   bathrooms       19765 non-null  float64       
 10  parking         17437 non-null  float64       
 11  floor_size      10977 non-null  float64       
 12  is_estate       20413 non-null  int64         
 13  has_security    20413 non-null  int64         
 14  is_modern       20413 non-null  int64         
dtypes: dateti

In [85]:
df.isnull().sum()

listing_number       0
scrape_at            0
price               43
title              862
type               862
suburb               0
city                 0
munis                0
bedrooms           779
bathrooms          648
parking           2976
floor_size        9436
is_estate            0
has_security         0
is_modern            0
dtype: int64

In [86]:
df['listing_number'].duplicated().sum()

np.int64(1521)

Removing all null prices rows as I cannot build a rent price model with null values

In [87]:
df = df.dropna(subset=['price'])

I will now only account for Residential properties i.e Apartments, Houses and Townhouse to make analysis uniform and easier. 

In [88]:
df['type'].value_counts()

type
Apartment              11717
House                   5521
Townhouse               2204
Commercial Property       52
Industrial Property       14
Name: count, dtype: int64

In [89]:
# drop listing number duplicate as I made a mistake when scraping
df = df.drop_duplicates(subset=['listing_number'])
# filter porperty type
df = df[df['type'].isin(['Apartment', 'House', 'Townhouse'])].copy()

In [90]:
# 2. Filter Outliers (Gauteng specific)
# Removing extreme outliers that break the scale
df = df[(df['price'] > 2500) & (df['price'] < 150000)]

In [91]:
# create a location tag with suburb, city because some suburbs have same names
df['location'] = (
    df['suburb'].str.lower().str.replace(' ', '_') + 
    '_' + 
    df['city'].str.lower().str.replace(' ', '_')
)

In [92]:
# Fill missing Beds/Baths with most common (mode)
df['bedrooms'] = df.groupby(['type'])['bedrooms'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else df['bedrooms'].mode()[0])
)
df['bathrooms'] = df.groupby(['type'])['bathrooms'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else df['bathrooms'].mode()[0])
)
df['parking'] = df['parking'].fillna(0) # Assume 0 if not stated


In [93]:
df['total_rooms'] = df['bedrooms'] + df['bathrooms']

In [94]:
# Smart Imputation for Floor Size
# 1️⃣ Most granular: location + total_rooms
mask = df['floor_size'].isna()
med_1 = df.groupby(['location', 'total_rooms'])['floor_size'].transform('median')
df.loc[mask, 'floor_size'] = med_1[mask]

# 2️⃣ Fallback: location only
mask = df['floor_size'].isna()
med_2 = df.groupby('location')['floor_size'].transform('median')
df.loc[mask, 'floor_size'] = med_2[mask]

# 3️⃣ Fallback: total_rooms only
mask = df['floor_size'].isna()
med_3 = df.groupby('total_rooms')['floor_size'].transform('median')
df.loc[mask, 'floor_size'] = med_3[mask]

# 4️⃣ Final fallback: global median
global_median = df['floor_size'].median()
df['floor_size'] = df['floor_size'].fillna(global_median)

In [95]:
# Target Encoding for suburb
# Why am I creating a suburb rank with the target variable 'price'?
suburb_means = df.groupby('location')['price'].mean()
df['suburb_rank'] = df['suburb'].map(suburb_means)
suburb_means

location
abbotsford_johannesburg    30000.000000
actonville_benoni           6000.000000
admin_katlehong             6000.000000
aeroton_johannesburg        6700.000000
agavia_krugersdorp          9231.250000
                               ...     
zandspruit_roodepoort      19325.000000
zola_soweto                 7600.000000
zwartkop_centurion         10351.750000
zwartkoppies_pretoria      12666.666667
zwavelpoort_ah_pretoria     7880.000000
Name: price, Length: 1107, dtype: float64

In [96]:
binary_cols = ['is_estate', 'has_security', 'is_modern']
# Calculate percentages (0 vs 1)
# normalize=True gives decimals, * 100 makes them percentages
stats = df[binary_cols].apply(lambda x: x.value_counts(normalize=True)).T * 100

# Rename columns for clarity
stats.columns = ['% No (0)', '% Yes (1)']

# Sort by the highest 'Yes' percentage to see which features are most common
stats = stats.sort_values('% Yes (1)', ascending=False)

print(stats.round(2))

              % No (0)  % Yes (1)
is_modern        74.30      25.70
has_security     76.44      23.56
is_estate        86.38      13.62


In [97]:
df.sort_values('scrape_at', ascending=False)

,listing_number,scrape_at,price,title,type,suburb,city,munis,bedrooms,bathrooms,parking,floor_size,is_estate,has_security,is_modern,location,total_rooms,suburb_rank
5944,116933533,2026-02-17 15:54:50.395613,3000.0,1 Bedroom House,House,andeon,pretoria,City of Tshwane,1.0,2.0,0.0,64.0,1,1,0,andeon_pretoria,3.0,NaN
5941,106937760,2026-02-17 15:54:50.393690,7000.0,2 Bedroom Apartment,Apartment,andeon,pretoria,City of Tshwane,2.0,1.0,0.0,64.0,0,0,1,andeon_pretoria,3.0,NaN
5939,116934159,2026-02-17 15:54:50.392205,12000.0,3 Bedroom Townhouse,Townhouse,andeon,pretoria,City of Tshwane,3.0,3.0,3.0,300.0,0,0,1,andeon_pretoria,6.0,NaN
6048,116728409,2026-02-17 02:54:27.197274,5600.0,2 Bedroom Apartment,Apartment,pyramid ah,wonderboom,City of Tshwane,2.0,2.0,4.0,65.0,0,0,0,pyramid_ah_wonderboom,4.0,NaN
6047,115921453,2026-02-17 02:54:27.196604,5600.0,2 Bedroom Apartment,Apartment,pyramid ah,wonderboom,City of Tshwane,2.0,2.0,4.0,65.0,0,0,0,pyramid_ah_wonderboom,4.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4,116775252,2026-02-15 23:51:52.554718,15000.0,3 Bedroom House,House,amandasig,akasia,City of Tshwane,3.0,3.0,4.0,500.0,1,1,1,amandasig_akasia,6.0,NaN
3,116794306,2026-02-15 23:51:52.554080,9400.0,3 Bedroom Townhouse,Townhouse,amandasig,akasia,City of Tshwane,3.0,2.5,3.0,400.0,0,0,0,amandasig_akasia,5.5,NaN
2,116915524,2026-02-15 23:51:52.553549,17070.0,5 Bedroom House,House,amandasig,akasia,City of Tshwane,5.0,4.0,2.0,400.0,0,0,0,amandasig_akasia,9.0,NaN
1,116914171,2026-02-15 23:51:52.552919,11000.0,3 Bedroom House,House,amandasig,akasia,City of Tshwane,3.0,2.0,4.0,300.0,1,1,0,amandasig_akasia,5.0,NaN


In [79]:
df.columns

Index(['listing_number', 'scrape_at', 'price', 'title', 'type', 'suburb',
       'city', 'munis', 'bedrooms', 'bathrooms', 'parking', 'floor_size',
       'is_estate', 'has_security', 'is_modern'],
      dtype='object')

In [ ]:
# # Create a 'Physical DNA' key
# # We round floor size to handle minor rounding differences (e.g., 80m2 vs 81m2)
# df['physical_dna'] = (
#     df['suburb'].astype(str) + "_" + 
#     df['bedrooms'].astype(str) + "_" + 
#     df['bathrooms'].astype(str)
#     # df['floor_size'].fillna(0).round(-1).astype(str) # Rounds 82 to 80, 87 to 90
# )

# # # Identify potential duplicates without deleting them
# # df['duplicate_count'] = df.groupby('property_fingerprint')['property_id'].transform('count')


In [ ]:
suburbs = df.groupby(['location']).agg(
    avg_rent=('price', 'mean'),
    median_rent=('price', 'median'),
    avg_floor_size=('floor_size', 'mean'),
).reset_index()
suburbs

,location,avg_rent,median_rent,avg_floor_size
0,abbotsford_johannesburg,30000.000000,30000.0,186.000
1,actonville_benoni,6000.000000,6000.0,110.000
2,admin_katlehong,6000.000000,6000.0,250.000
3,aeroton_johannesburg,6700.000000,6950.0,49.000
4,agavia_krugersdorp,9231.250000,10000.0,129.625
...,...,...,...,...
1102,zandspruit_roodepoort,19325.000000,19250.0,237.500
1103,zola_soweto,7600.000000,7600.0,219.000
1104,zwartkop_centurion,10351.750000,8350.0,275.100
1105,zwartkoppies_pretoria,12666.666667,11000.0,45.000


In [99]:
df['suburb_avg_rent'] = df['location'].map(suburb_means)

In [102]:
df.sample(5)

,listing_number,scrape_at,price,title,type,suburb,city,munis,bedrooms,bathrooms,parking,floor_size,is_estate,has_security,is_modern,location,total_rooms,suburb_rank,suburb_avg_rent
317,116423468,2026-02-17 02:23:43.838703,7700.0,2 Bedroom Townhouse,Townhouse,three rivers,vereeniging,Sedibeng,2.0,1.0,2.0,115.0,0,1,0,three_rivers_vereeniging,3.0,NaN,9210.769231
831,116151719,2026-02-16 07:01:19.890957,4950.0,4 Bedroom Apartment,Apartment,braampark,johannesburg,City of Johannesburg,4.0,2.5,1.0,377.0,0,0,0,braampark_johannesburg,6.5,NaN,4950.000000
5266,116383037,2026-02-16 17:27:47.071073,11500.0,1.5 Bedroom Apartment,Apartment,sunnyside,pretoria,City of Tshwane,1.5,1.5,1.0,8.0,0,0,1,sunnyside_pretoria,3.0,NaN,5222.391753
10136,116828306,2026-02-16 22:33:05.706767,21000.0,1 Bedroom Apartment,Apartment,sandton central,sandton,City of Johannesburg,1.0,1.0,2.0,57.0,0,0,0,sandton_central_sandton,2.0,NaN,43419.095406
1930,116723918,2026-02-16 14:41:09.899313,14999.0,2 Bedroom Apartment,Apartment,brooklyn,pretoria,City of Tshwane,2.0,2.0,2.0,68.5,0,0,1,brooklyn_pretoria,4.0,NaN,15020.875536


In [107]:
file_path = "../data/4_feature_store/suburb_rent_stats.parquet"

df_c = pd.read_parquet(file_path)
df_c

,price
0,30000.000000
1,6000.000000
2,6000.000000
3,6460.000000
4,9231.250000
...,...
1102,19325.000000
1103,9066.666667
1104,10351.750000
1105,11250.000000


In [49]:
path = "../data/0_metadata/suburb_lookup.parquet"

lookup = pd.read_parquet(path)


In [51]:
lookup[lookup['last_scraped']<'2026-02-15']

,suburb_id,suburb_name,city_id,city_name,province_id,province_name,municipality,last_scraped
1131,4115,andeon,1,pretoria,1,gauteng,City of Tshwane,2026-02-03
